In [1]:
import os
project_root = "/Users/katherinele/Desktop/my_emotune"

os.chdir(project_root)

print(os.getcwd())
print(os.listdir())

/Users/katherinele/Desktop/my_emotune
['frontend', 'requirements.txt', 'backend', 'README.md', '.gitignore', '.env', 'venv', '.git', 'data', 'notebooks', 'src']


In [22]:
from src.recommender.similarity_engine import get_similar_songs

recommendations = get_similar_songs(
    track_id="6Yj8kVuVR3UPxx9r5eFEoV",
    top_k=10,
)

print(recommendations[
    ["track_name", "track_artist", "similarity", "explanation"]
].to_string(index=False))

         track_name      track_artist  similarity                                                             explanation
      Dancing Queen Girls' Generation    0.951987 Similar because of instrumentalness, emotional positivity, acousticness
         Short Hair               AOA    0.908227          Similar because of instrumentalness, speechiness, danceability
              Alone            SISTAR    0.901617             Similar because of instrumentalness, danceability, loudness
            Twinkle Girls' Generation    0.899203  Similar because of instrumentalness, emotional positivity, speechiness
         Miss Right          TEEN TOP    0.896408        Similar because of instrumentalness, tempo, emotional positivity
    SAY YOU LOVE ME             TWICE    0.878547       Similar because of instrumentalness, emotional positivity, energy
              Shoes                IU    0.863760          Similar because of instrumentalness, danceability, speechiness
            Healing     

In [3]:
from src.recommender.similarity_engine import _get_songs
from src.recommender.feature_builder import AUDIO_FEATURES

songs = _get_songs()

selected_songs = songs[
    songs["track_id"].isin([
        "405SQUJdQut02dxtuQ0CZ3",
    ])
    |
    songs["track_name"].str.lower().isin([
        "last dance",
        "above the time",
    ])
]

print(
    selected_songs[
        ["track_name", "track_artist"] + AUDIO_FEATURES
    ].to_string(index=False)
)

    track_name      track_artist  danceability  energy  loudness  speechiness  acousticness  instrumentalness  liveness  valence   tempo
    Lion Heart Girls' Generation         0.754   0.788    -3.175       0.0322         0.129           0.00274    0.0775    0.961 124.992
    LAST DANCE           BIGBANG         0.614   0.666    -3.087       0.0541         0.455           0.00000    0.2300    0.299 143.858
above the time                IU         0.471   0.548    -4.301       0.0369         0.742           0.00000    0.0896    0.177 147.961


In [23]:
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import (
    cosine_similarity,
    euclidean_distances,
)

from src.recommender.similarity_engine import (
    _get_songs,
    _get_feature_data,
)

# Load songs and feature matrix
songs = _get_songs()
matrix, scaler = _get_feature_data("default")

# Find Lion Heart
track_id = "6Yj8kVuVR3UPxx9r5eFEoV"

song_index = songs.index[
    songs["track_id"] == track_id
][0]

# Calculate cosine similarity
cosine_scores = cosine_similarity(
    matrix[song_index].reshape(1, -1),
    matrix,
)[0]

# Calculate Euclidean distance
distances = euclidean_distances(
    matrix[song_index].reshape(1, -1),
    matrix,
)[0]

# Create comparison table
comparison = songs[
    ["track_name", "track_artist"]
].copy()

comparison["cosine_similarity"] = cosine_scores
comparison["euclidean_distance"] = distances

# Exclude reference song
comparison = comparison.drop(index=song_index)

# Get top 10 from cosine
cosine_top10 = comparison.sort_values(
    "cosine_similarity",
    ascending=False,
).head(10)

# Get top 10 from Euclidean
euclidean_top10 = comparison.sort_values(
    "euclidean_distance",
    ascending=True,
).head(10)

print("COSINE SIMILARITY")
print(cosine_top10.to_string(index=False))

print("\nEUCLIDEAN DISTANCE")
print(euclidean_top10.to_string(index=False))

COSINE SIMILARITY
     track_name      track_artist  cosine_similarity  euclidean_distance
  Dancing Queen Girls' Generation           0.951987            0.866463
     Short Hair               AOA           0.908227            1.096898
          Alone            SISTAR           0.901617            1.138709
        Twinkle Girls' Generation           0.899203            1.226943
     Miss Right          TEEN TOP           0.896408            1.190141
SAY YOU LOVE ME             TWICE           0.878547            1.297918
          Shoes                IU           0.863760            1.585880
    Kissing You Girls' Generation           0.858621            1.409441
        Healing         SEVENTEEN           0.856177            1.557646
      Baby Baby Girls' Generation           0.835607            1.437632

EUCLIDEAN DISTANCE
                  track_name      track_artist  cosine_similarity  euclidean_distance
               Dancing Queen Girls' Generation           0.951987        

In [7]:
print("DataFrame Lion Heart:")
print(songs.iloc[lion_index][["track_name", "track_artist"]])

print("\nDataFrame LAST DANCE:")
print(songs.iloc[last_dance_index][["track_name", "track_artist"]])

print("\nMatrix shape:", matrix.shape)
print("Songs shape:", songs.shape)

DataFrame Lion Heart:
track_name             Lion Heart
track_artist    Girls' Generation
Name: 367, dtype: object

DataFrame LAST DANCE:
track_name      LAST DANCE
track_artist       BIGBANG
Name: 394, dtype: object

Matrix shape: (1041, 9)
Songs shape: (1041, 21)


In [8]:
from src.recommender.similarity_engine import (
    _get_songs,
    _get_feature_data,
)

_get_songs.cache_clear()
_get_feature_data.cache_clear()

In [13]:
songs = _get_songs()
matrix, scaler = _get_feature_data("default")

lion_index = songs.index[
    songs["track_id"] == "405SQUJdQut02dxtuQ0CZ3"
][0]

last_dance_index = songs.index[
    songs["track_name"].str.lower() == "last dance"
][0]

print(songs.iloc[lion_index][["track_name", "track_artist"]])
print(songs.iloc[last_dance_index][["track_name", "track_artist"]])

track_name      LAST DANCE
track_artist       BIGBANG
Name: 422, dtype: object
track_name      WHISTLE - Japanese Version
track_artist                     BLACKPINK
Name: 450, dtype: object


In [9]:
recommendations = get_similar_songs(
    track_id="405SQUJdQut02dxtuQ0CZ3",
    top_k=10,
)

print(
    recommendations[
        ["track_name", "track_artist", "similarity", "explanation"]
    ].to_string(index=False)
)

       track_name      track_artist  similarity                                                                             explanation
            3.6.5               EXO    0.973729                             Similar because of instrumentalness, loudness, danceability
         Bad Girl             Beast    0.968663            Similar because of tempo, instrumentalness, live-performance characteristics
        Lucky One               EXO    0.967418           Similar because of energy, instrumentalness, live-performance characteristics
            PARTY Girls' Generation    0.965592     Similar because of danceability, live-performance characteristics, instrumentalness
      Chewing Gum         NCT DREAM    0.929276           Similar because of live-performance characteristics, instrumentalness, energy
What's your name?           4Minute    0.908856         Similar because of instrumentalness, live-performance characteristics, loudness
        SEXY LOVE             T-ARA    0.906879 

In [10]:
import pandas as pd
import numpy as np

songs = _get_songs()
matrix, scaler = _get_feature_data("default")

print("Index is sequential:",
      songs.index.equals(pd.RangeIndex(len(songs))))

print("Matrix rows:", matrix.shape[0])
print("DataFrame rows:", len(songs))

lion_pos = np.flatnonzero(
    songs["track_id"].eq("405SQUJdQut02dxtuQ0CZ3").to_numpy()
)[0]

last_pos = np.flatnonzero(
    songs["track_name"].str.lower().eq("last dance").to_numpy()
)[0]

print("\nLion Heart position:", lion_pos)
print(songs.iloc[lion_pos][["track_name", "track_artist"]])

print("\nLAST DANCE position:", last_pos)
print(songs.iloc[last_pos][["track_name", "track_artist"]])

print("\nDirect cosine similarity:",
      cosine_similarity(
          matrix[lion_pos].reshape(1, -1),
          matrix[last_pos].reshape(1, -1)
      )[0][0])

Index is sequential: True
Matrix rows: 1041
DataFrame rows: 1041

Lion Heart position: 367
track_name             Lion Heart
track_artist    Girls' Generation
Name: 367, dtype: object

LAST DANCE position: 394
track_name      LAST DANCE
track_artist       BIGBANG
Name: 394, dtype: object

Direct cosine similarity: -0.4731599232661959


In [2]:
from pathlib import Path
import sys

# Go up one level from the notebooks folder
PROJECT_ROOT = Path.cwd().parent

# Tell Python to look in the project root
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Root folders:", [p.name for p in PROJECT_ROOT.iterdir()])

Project root: /Users/katherinele/Desktop/my_emotune
Root folders: ['frontend', 'requirements.txt', 'backend', 'README.md', '.gitignore', '.env', 'venv', '.git', 'data', 'notebooks', 'src']


In [3]:
from backend.main import TextRecommendationRequest
from pydantic import ValidationError

In [5]:
request = TextRecommendationRequest(
    text="I've had a stressful day.",
    current_mood="stressed",
    desired_mood="relaxed",
    intensity=4,
    top_n=5
)

print(request.model_dump())

{'text': "I've had a stressful day.", 'mood': None, 'current_mood': 'stressed', 'desired_mood': 'relaxed', 'intensity': 4, 'top_n': 5}


In [6]:
request = TextRecommendationRequest(
    mood="happy",
    top_n=5
)

print(request)

text=None mood='happy' current_mood=None desired_mood=None intensity=3 top_n=5


In [7]:
try:
    request = TextRecommendationRequest(
        text="I've had a stressful day.",
        top_n=5
    )
except ValidationError as e:
    print(e)

1 validation error for TextRecommendationRequest
  Value error, Provide either a mood or text with a desired mood. [type=value_error, input_value={'text': "I've had a stressful day.", 'top_n': 5}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.5/v/value_error


In [8]:
try:
    request = TextRecommendationRequest(
        mood="happy",
        intensity=10
    )
except ValidationError as e:
    print(e)

1 validation error for TextRecommendationRequest
intensity
  Input should be less than or equal to 5 [type=less_than_equal, input_value=10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.5/v/less_than_equal


In [9]:
test_cases = [
    {
        "name": "Valid emotional context",
        "data": {
            "text": "I had a stressful day.",
            "current_mood": "stressed",
            "desired_mood": "relaxed",
            "intensity": 4,
            "top_n": 5
        },
        "should_pass": True
    },
    {
        "name": "Valid direct mood",
        "data": {
            "mood": "happy",
            "top_n": 5
        },
        "should_pass": True
    },
    {
        "name": "Missing desired mood",
        "data": {
            "text": "I had a stressful day."
        },
        "should_pass": False
    },
    {
        "name": "Invalid intensity",
        "data": {
            "mood": "happy",
            "intensity": 10
        },
        "should_pass": False
    }
]

for test in test_cases:
    try:
        TextRecommendationRequest(**test["data"])
        passed = True
    except ValidationError:
        passed = False

    result = "PASS" if passed == test["should_pass"] else "FAIL"

    print(f'{result}: {test["name"]}')

PASS: Valid emotional context
PASS: Valid direct mood
PASS: Missing desired mood
PASS: Invalid intensity


In [4]:
from fastapi.testclient import TestClient
from backend.main import app

client = TestClient(app)

In [7]:
response = client.post(
    "/api/recommend/text",
    json={
        "mood": "happy",
        "top_n": 5
    }
)

print("Status:", response.status_code)
print(response.json())

INFO:httpx:HTTP Request: POST http://testserver/api/recommend/text "HTTP/1.1 200 OK"


Status: 200
{'emotions': [{'name': 'happy', 'score': 1.0}], 'top_emotion': 'happy', 'top_score': 1.0, 'recommendations': [{'track_id': '3IQ9FJ2cMVQtger5ZhgCaj', 'track_name': 'Devil', 'track_artist': 'SUPER JUNIOR', 'similarity': 0.5374032694585125, 'danceability': 0.773, 'energy': 0.865, 'loudness': -3.027, 'speechiness': 0.114, 'acousticness': 0.133, 'instrumentalness': 0.0, 'liveness': 0.052, 'valence': 0.968, 'tempo': 145.003}, {'track_id': '3E29VjkpQlzslDyCqYewDm', 'track_name': '3.6.5', 'track_artist': 'EXO', 'similarity': 0.5087708765912058, 'danceability': 0.74, 'energy': 0.83, 'loudness': -3.051, 'speechiness': 0.0434, 'acousticness': 0.0678, 'instrumentalness': 0.0, 'liveness': 0.0534, 'valence': 0.93, 'tempo': 131.042}, {'track_id': '0qD5kstOSUeLUQgW3cuhcD', 'track_name': 'To My Boyfriend', 'track_artist': 'Fin.K.L', 'similarity': 0.501099500560676, 'danceability': 0.773, 'energy': 0.857, 'loudness': -5.884, 'speechiness': 0.032, 'acousticness': 0.0112, 'instrumentalness': 0

In [8]:
response = client.post(
    "/api/recommend/text",
    json={
        "text": "I've had a stressful day.",
        "current_mood": "stressed",
        "desired_mood": "relaxed",
        "intensity": 4,
        "top_n": 5
    }
)

print("Status:", response.status_code)
print(response.json())

ERROR:emotion_service:Hume status: 403
ERROR:emotion_service:Hume response: 
                {"code":403, "message":"The Expression Measurement API has been discontinued and is no longer available."}
            
INFO:httpx:HTTP Request: POST http://testserver/api/recommend/text "HTTP/1.1 502 Bad Gateway"


Status: 502
{'detail': 'Hume API request failed: 403 Client Error: Forbidden for url: https://api.hume.ai/v0/batch/jobs'}


In [11]:
import httpx
import starlette
import fastapi
import sys

print("Python executable:", sys.executable)
print("HTTPX:", httpx.__version__)
print("Starlette:", starlette.__version__)
print("FastAPI:", fastapi.__version__)

Python executable: /usr/local/bin/python3
HTTPX: 0.28.1
Starlette: 0.27.0
FastAPI: 0.104.1


In [12]:
%pip install "httpx==0.27.2"

  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-core 0.3.78 requires pydantic<3.0.0,>=2.7.4, but you have pydantic 2.5.0 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
